## Data Pipeline Steps - Roadmap
1. Data Extraction
2. Quality Assessment  
3. Data Cleaning
4. Business Validation
5. Feature Engineering
6. Analysis & Insights

### Install and Import Library

In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# import libraries 
import pandas as pd 

### Data Extraction

In [3]:
# Define proper data types for columns like phone numbers
dtype_dict = {
    'phone_number': 'str',
    'customer_id': 'str',
    'tower_id': 'str',
    'call_id': 'str'
}

In [4]:
# read in data
df_cdr = pd.read_csv(r'C:\Users\Iyanujesu\Downloads\MTN_pandas_data_engineering\data\raw\MTN_cdr_data.csv', dtype=dtype_dict)

In [5]:
# see top rows
df_cdr.head()

,call_id,customer_id,phone_number,tower_id,call_type,call_duration_seconds,data_usage_mb,signal_strength_dbm,call_timestamp,call_success,revenue_naira,network_type,roaming
0,86634543-444c-4f38-aa1d-70df574593fa,CU000151,08169541064,INVALID_TW9116,SMS,2532.0,4666.74,-85.0,2024-04-30 04:30:43,True,842.74,4G,True
1,fe36e9b9-f944-4d9a-83b2-a1796fc5074c,CU000307,07027999942,TW0040,data,NaN,3281.08,-74.0,2024-04-08 10:37:21,True,126.07,5G,False
2,534acb40-196b-469b-8b20-dbbe23a6e898,CU001104,07011766356,TW0128,Voice,5506.0,3967.93,NaN,2024-05-04 16:11:44,0,400.31,5G,0
3,04b2ab83-5325-4dd0-9a46-6d767439f5a2,CU000869,09059923201,NaN,Voice,4610.0,4734.73,-113.0,2024-05-24 14:01:58,False,956.19,3g,True
4,7775e1be-3638-4d37-8ae9-e91fd5a966e5,CU000518,08091140319,TW0150,SMS,271.0,2995.66,-106.0,2024-04-07 20:54:53,False,740.63,3G,1


### Understanding our data structure

In [6]:
# shape of our dataset
df_cdr.shape

(5137, 13)

In [7]:
df_cdr.columns

Index(['call_id', 'customer_id', 'phone_number', 'tower_id', 'call_type',
       'call_duration_seconds', 'data_usage_mb', 'signal_strength_dbm',
       'call_timestamp', 'call_success', 'revenue_naira', 'network_type',
       'roaming'],
      dtype='object')

In [8]:
df_cdr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5137 entries, 0 to 5136
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   call_id                5137 non-null   object 
 1   customer_id            4712 non-null   object 
 2   phone_number           4711 non-null   object 
 3   tower_id               4706 non-null   object 
 4   call_type              5112 non-null   object 
 5   call_duration_seconds  4710 non-null   float64
 6   data_usage_mb          4703 non-null   float64
 7   signal_strength_dbm    4706 non-null   float64
 8   call_timestamp         5112 non-null   object 
 9   call_success           5112 non-null   object 
 10  revenue_naira          5112 non-null   float64
 11  network_type           5112 non-null   object 
 12  roaming                5112 non-null   object 
dtypes: float64(4), object(9)
memory usage: 521.9+ KB


### Data Quality Assessment

#### Check for missing values

In [9]:
df_cdr.isnull().sum()

call_id                    0
customer_id              425
phone_number             426
tower_id                 431
call_type                 25
call_duration_seconds    427
data_usage_mb            434
signal_strength_dbm      431
call_timestamp            25
call_success              25
revenue_naira             25
network_type              25
roaming                   25
dtype: int64

In [10]:
# missing values 
overall_missing_percentage = df_cdr.isnull().sum().sum() / df_cdr.size * 100
print(f'overall_percent_value: {overall_missing_percentage:.2f}%')

overall_percent_value: 4.08%


### Check for duplicates

In [11]:
duplicates = df_cdr.duplicated().sum()
duplicates

np.int64(112)

In [12]:
if duplicates > 0:
    duplicated_rows = df_cdr[df_cdr.duplicated(keep=False)]
    print(duplicated_rows.head())

                                  call_id customer_id phone_number tower_id  \
108  45191a1c-1767-41fc-a3d3-31883483b53d    CU000001  09058984075      NaN   
109  45191a1c-1767-41fc-a3d3-31883483b53d    CU000001  09058984075      NaN   
125  d3b8e673-f066-4f99-b66b-effe4feaf277    CU000819  09039727843      NaN   
126  d3b8e673-f066-4f99-b66b-effe4feaf277    CU000819  09039727843      NaN   
177  a082047a-8441-47de-9258-d1c9bfa52bd9    CU001447  08099575199      NaN   

    call_type  call_duration_seconds  data_usage_mb  signal_strength_dbm  \
108     Voice                 1502.0        2063.49                -41.0   
109     Voice                 1502.0        2063.49                -41.0   
125       SMS                   26.0         146.76                -45.0   
126       SMS                   26.0         146.76                -45.0   
177      data                 3494.0        4549.78                -51.0   

          call_timestamp call_success  revenue_naira network_type ro

### Check for inconsistent/unusal values

In [13]:
df_cdr['call_success'].value_counts()

call_success
N        678
0        669
False    664
Y        643
1        633
Yes      632
No       605
True     588
Name: count, dtype: int64

In [14]:
df_cdr['call_type'].value_counts()

call_type
sms      894
voice    866
data     846
Data     845
SMS      832
Voice    829
Name: count, dtype: int64

In [15]:
df_cdr['network_type'].value_counts()

network_type
5G    907
4G    858
4g    854
2G    846
3G    825
3g    822
Name: count, dtype: int64

In [16]:
inconsistent_columns = ['call_success', 'roaming', 'call_type', 'network_type']

In [17]:
# loop through each columns to check
for col in inconsistent_columns:
    if col in df_cdr.columns:
        print(df_cdr[col].value_counts())


call_success
N        678
0        669
False    664
Y        643
1        633
Yes      632
No       605
True     588
Name: count, dtype: int64
roaming
No       881
True     872
False    868
1        847
0        834
Yes      810
Name: count, dtype: int64
call_type
sms      894
voice    866
data     846
Data     845
SMS      832
Voice    829
Name: count, dtype: int64
network_type
5G    907
4G    858
4g    854
2G    846
3G    825
3g    822
Name: count, dtype: int64


In [18]:
# negative call durations
negative_durations = df_cdr[df_cdr['call_duration_seconds'] < 0]
len(negative_durations)

87

In [19]:
# incorrect phone number format
phone_lenghts = df_cdr['phone_number'].str.len()
phone_lenghts.value_counts()

phone_number
11.0    4253
9.0      233
10.0     163
14.0      62
Name: count, dtype: int64

In [20]:
phone_number = df_cdr['phone_number'].head(10)
phone_number

0    08169541064
1    07027999942
2    07011766356
3    09059923201
4    08091140319
5    09052381677
6    07086288400
7      080864810
8    08021197996
9    09067384306
Name: phone_number, dtype: object

In [21]:
# to see each number
phone_lenghts = df_cdr[df_cdr['phone_number'].str.len() == 14]
print(phone_lenghts['phone_number'])


17      +2348131529592
250     +2348036952196
251     +2348068569127
283     +2348138291940
454     +2348139816744
             ...      
4646    +2348032065738
4683    +2348139816744
4832    +2348136526890
5028    +2348035732779
5047    +2348138291940
Name: phone_number, Length: 62, dtype: object


# Data Cleaning and Transformation

In [22]:
df_clean = df_cdr.copy()

1. Incorrect data type
2. Missing values 
3. Handle Duplicates
4. Standardize text formats(Inconsistent values)

In [23]:
# Convert to correct data type
df_clean['call_timestamp'] = pd.to_datetime(df_clean['call_timestamp'])

### handle missing values 
1. drop rows in critical columns - phone numbers, customer id
2. fill null values


In [24]:
# remove/drop rows for customer id and phone numbers
df_clean = df_clean.dropna(subset=['customer_id', 'phone_number'])

In [25]:
df_clean.head()

,call_id,customer_id,phone_number,tower_id,call_type,call_duration_seconds,data_usage_mb,signal_strength_dbm,call_timestamp,call_success,revenue_naira,network_type,roaming
0,86634543-444c-4f38-aa1d-70df574593fa,CU000151,08169541064,INVALID_TW9116,SMS,2532.0,4666.74,-85.0,2024-04-30 04:30:43,True,842.74,4G,True
1,fe36e9b9-f944-4d9a-83b2-a1796fc5074c,CU000307,07027999942,TW0040,data,NaN,3281.08,-74.0,2024-04-08 10:37:21,True,126.07,5G,False
2,534acb40-196b-469b-8b20-dbbe23a6e898,CU001104,07011766356,TW0128,Voice,5506.0,3967.93,NaN,2024-05-04 16:11:44,0,400.31,5G,0
3,04b2ab83-5325-4dd0-9a46-6d767439f5a2,CU000869,09059923201,NaN,Voice,4610.0,4734.73,-113.0,2024-05-24 14:01:58,False,956.19,3g,True
4,7775e1be-3638-4d37-8ae9-e91fd5a966e5,CU000518,08091140319,TW0150,SMS,271.0,2995.66,-106.0,2024-04-07 20:54:53,False,740.63,3G,1


In [26]:
df_clean.isnull().sum()

call_id                    0
customer_id                0
phone_number               0
tower_id                 356
call_type                  0
call_duration_seconds    343
data_usage_mb            353
signal_strength_dbm      340
call_timestamp             0
call_success               0
revenue_naira              0
network_type               0
roaming                    0
dtype: int64

In [27]:
# fill null values with fillna
fill_values = {
    'call_duration_seconds': 0,
    'data_usage_mb': 0,
    'tower_id': 'UNKNOWN',
    'signal_strength_dbm': df_clean['signal_strength_dbm'].median()
}

df_clean = df_clean.fillna(fill_values)

In [28]:
df_clean.isnull().sum()

call_id                  0
customer_id              0
phone_number             0
tower_id                 0
call_type                0
call_duration_seconds    0
data_usage_mb            0
signal_strength_dbm      0
call_timestamp           0
call_success             0
revenue_naira            0
network_type             0
roaming                  0
dtype: int64

In [29]:
# handle duplicates
df_clean = df_clean.drop_duplicates()

In [30]:
#check
duplicates = df_clean.duplicated().sum()
duplicates

np.int64(0)

### Stardardize Text formats

In [31]:
# standardize boolean columns
def standardize_boolean(value):
    if pd.isna(value):
        return None
    value_str = str(value).lower()
    if value_str in ['yes', 'true', '1', 'y']:
        return True
    elif value_str in ['no', 'false', '0', 'n']:
        return False
    else:
        return None

In [32]:
df_clean['call_success'] = df_clean['call_success'].apply(standardize_boolean)
df_clean['roaming'] = df_clean['roaming'].apply(standardize_boolean)


In [33]:
df_clean['roaming'].value_counts()

roaming
False    2130
True     2119
Name: count, dtype: int64

In [ ]:
# standardize call type and network type
def standardize_text_formats(df):
    
    # Standardize all_type to Title Case
    if 'call_type' in df_clean.columns:
        df_clean['call_type'] = df_clean['call_type'].str.lower().str.title()
    
    # Standardize network_type to uppercase
    if 'network_type' in df_clean.columns:
        df_clean['network_type'] = df_clean['network_type'].str.upper()
    
    return df_clean



df_clean = standardize_text_formats(df_clean)


### Convert phone numbers to correct format

In [48]:
def convert_phone_format(phone):
# check that the phone is a string
    phone_str = str(phone)

    # replace + with empty ''
    cleaned_phone = ''.join(filter(str.isdigit, phone_str))

    # convert 234 to 0
    if cleaned_phone.startswith('234'):
        # return 0 + the remaining number
        return '0' + cleaned_phone[3:]
    else:
        return cleaned_phone

In [49]:
df_clean['phone_number'] = df_clean['phone_number'].apply(convert_phone_format)

In [50]:
# check phone number format
phone_lenghts = df_clean['phone_number'].str.len()
phone_lenghts.value_counts()

phone_number
11    3885
9      213
10     151
Name: count, dtype: int64

## Business rule validation

#### Phone number validation

In [51]:
# phone number validation
def validate_phone_format(phone):
    # check that the phone is a string
    phone_str = str(phone)
    
    # check 1
    # make sure the numbers are in digita - no symbols, letters
    if not phone_str.isdigit():
        return False
    #check 2
    # number should be 11
    if len(phone_str) != 11:
        return False
    
    return True
    

In [52]:
df_clean['phone_valid'] =  df_clean['phone_number'].apply(validate_phone_format)

In [53]:
df_clean.head()

,call_id,customer_id,phone_number,tower_id,call_type,call_duration_seconds,data_usage_mb,signal_strength_dbm,call_timestamp,call_success,revenue_naira,network_type,roaming,phone_valid
0,86634543-444c-4f38-aa1d-70df574593fa,CU000151,08169541064,INVALID_TW9116,Sms,2532.0,4666.74,-85.0,2024-04-30 04:30:43,True,842.74,4G,True,True
1,fe36e9b9-f944-4d9a-83b2-a1796fc5074c,CU000307,07027999942,TW0040,Data,0.0,3281.08,-74.0,2024-04-08 10:37:21,True,126.07,5G,False,True
2,534acb40-196b-469b-8b20-dbbe23a6e898,CU001104,07011766356,TW0128,Voice,5506.0,3967.93,-75.0,2024-05-04 16:11:44,False,400.31,5G,False,True
3,04b2ab83-5325-4dd0-9a46-6d767439f5a2,CU000869,09059923201,UNKNOWN,Voice,4610.0,4734.73,-113.0,2024-05-24 14:01:58,False,956.19,3G,True,True
4,7775e1be-3638-4d37-8ae9-e91fd5a966e5,CU000518,08091140319,TW0150,Sms,271.0,2995.66,-106.0,2024-04-07 20:54:53,False,740.63,3G,True,True


In [54]:
total_phones = df_clean['phone_number'].notna().sum()
valid_phones = df_clean['phone_valid'].sum()

invalid_phones = total_phones - valid_phones

invalid_phones


np.int64(364)

In [55]:
df_clean['phone_valid'].value_counts()


phone_valid
True     3885
False     364
Name: count, dtype: int64

### Validating Signal Strength

Signal strength should be between -120 and -30 dBm

In [56]:
# validate signal strength

def validate_signal_strength(signal):
    if pd.isna(signal):
        return None
    return -120 <= signal <= -30

In [57]:
df_clean['signal_valid'] = df_clean['signal_strength_dbm'].apply(validate_signal_strength)
df_clean['signal_valid'].value_counts()

signal_valid
True     4126
False     123
Name: count, dtype: int64

In [58]:
df_clean['signal_valid'].astype(bool)

0       True
1       True
2       True
3       True
4       True
        ... 
5106    True
5107    True
5108    True
5109    True
5110    True
Name: signal_valid, Length: 4249, dtype: bool

## Feature Engineering 

In [59]:
df_clean.head()

,call_id,customer_id,phone_number,tower_id,call_type,call_duration_seconds,data_usage_mb,signal_strength_dbm,call_timestamp,call_success,revenue_naira,network_type,roaming,phone_valid,signal_valid
0,86634543-444c-4f38-aa1d-70df574593fa,CU000151,08169541064,INVALID_TW9116,Sms,2532.0,4666.74,-85.0,2024-04-30 04:30:43,True,842.74,4G,True,True,True
1,fe36e9b9-f944-4d9a-83b2-a1796fc5074c,CU000307,07027999942,TW0040,Data,0.0,3281.08,-74.0,2024-04-08 10:37:21,True,126.07,5G,False,True,True
2,534acb40-196b-469b-8b20-dbbe23a6e898,CU001104,07011766356,TW0128,Voice,5506.0,3967.93,-75.0,2024-05-04 16:11:44,False,400.31,5G,False,True,True
3,04b2ab83-5325-4dd0-9a46-6d767439f5a2,CU000869,09059923201,UNKNOWN,Voice,4610.0,4734.73,-113.0,2024-05-24 14:01:58,False,956.19,3G,True,True,True
4,7775e1be-3638-4d37-8ae9-e91fd5a966e5,CU000518,08091140319,TW0150,Sms,271.0,2995.66,-106.0,2024-04-07 20:54:53,False,740.63,3G,True,True,True


#### New columns
- call duration in minutes 
- date features
- revenue features
- signal strength feature

In [ ]:
# create call duration in minutes column
df_clean['call_duration_minutes'] = (df_clean['call_duration_seconds'] / 60).round(2)
df_clean['call_duration_minutes'].head()

0    42.20
1     0.00
2    91.77
3    76.83
4     4.52
Name: call_duration_minutes, dtype: float64

In [61]:
# create date features column

df_clean['call_month'] = df_clean['call_timestamp'].dt.month
df_clean['call_month']

0       4
1       4
2       5
3       5
4       4
       ..
5106    5
5107    5
5108    5
5109    6
5110    3
Name: call_month, Length: 4249, dtype: int32

In [62]:
df_clean['call_timestamp'].dt.month_name()

0       April
1       April
2         May
3         May
4       April
        ...  
5106      May
5107      May
5108      May
5109     June
5110    March
Name: call_timestamp, Length: 4249, dtype: object

In [63]:
# extract call hour and day of week
df_clean['call_hour'] = df_clean['call_timestamp'].dt.hour
df_clean['day_of_week'] = df_clean['call_timestamp'].dt.day_name()


In [64]:
# create new column for high paying customers
df_clean['revenue_naira'].quantile(0.8)

np.float64(794.826)

In [65]:
df_clean['high_value_customers'] = df_clean['revenue_naira'] > df_clean['revenue_naira'].quantile(0.8)
df_clean.head()

,call_id,customer_id,phone_number,tower_id,call_type,call_duration_seconds,data_usage_mb,signal_strength_dbm,call_timestamp,call_success,revenue_naira,network_type,roaming,phone_valid,signal_valid,call_duration_minutes,call_month,call_hour,day_of_week,high_value_customers
0,86634543-444c-4f38-aa1d-70df574593fa,CU000151,08169541064,INVALID_TW9116,Sms,2532.0,4666.74,-85.0,2024-04-30 04:30:43,True,842.74,4G,True,True,True,42.20,4,4,Tuesday,True
1,fe36e9b9-f944-4d9a-83b2-a1796fc5074c,CU000307,07027999942,TW0040,Data,0.0,3281.08,-74.0,2024-04-08 10:37:21,True,126.07,5G,False,True,True,0.00,4,10,Monday,False
2,534acb40-196b-469b-8b20-dbbe23a6e898,CU001104,07011766356,TW0128,Voice,5506.0,3967.93,-75.0,2024-05-04 16:11:44,False,400.31,5G,False,True,True,91.77,5,16,Saturday,False
3,04b2ab83-5325-4dd0-9a46-6d767439f5a2,CU000869,09059923201,UNKNOWN,Voice,4610.0,4734.73,-113.0,2024-05-24 14:01:58,False,956.19,3G,True,True,True,76.83,5,14,Friday,True
4,7775e1be-3638-4d37-8ae9-e91fd5a966e5,CU000518,08091140319,TW0150,Sms,271.0,2995.66,-106.0,2024-04-07 20:54:53,False,740.63,3G,True,True,True,4.52,4,20,Sunday,False


In [66]:
# create signal strength column -30 and -120
def categorize_signal_strength(signal):
    if pd.isna(signal):
        return 'Unknown'
    elif signal >= -40:
        return 'Execellent'
    elif signal >= -70:
        return 'Good'
    elif signal >= -100:
        return 'Fair'
    else:
        return 'Poor'
    
df_clean['signal_quality'] = df_clean['signal_strength_dbm'].apply(categorize_signal_strength)

In [67]:
df_clean.head()

,call_id,customer_id,phone_number,tower_id,call_type,call_duration_seconds,data_usage_mb,signal_strength_dbm,call_timestamp,call_success,...,network_type,roaming,phone_valid,signal_valid,call_duration_minutes,call_month,call_hour,day_of_week,high_value_customers,signal_quality
0,86634543-444c-4f38-aa1d-70df574593fa,CU000151,08169541064,INVALID_TW9116,Sms,2532.0,4666.74,-85.0,2024-04-30 04:30:43,True,...,4G,True,True,True,42.20,4,4,Tuesday,True,Fair
1,fe36e9b9-f944-4d9a-83b2-a1796fc5074c,CU000307,07027999942,TW0040,Data,0.0,3281.08,-74.0,2024-04-08 10:37:21,True,...,5G,False,True,True,0.00,4,10,Monday,False,Fair
2,534acb40-196b-469b-8b20-dbbe23a6e898,CU001104,07011766356,TW0128,Voice,5506.0,3967.93,-75.0,2024-05-04 16:11:44,False,...,5G,False,True,True,91.77,5,16,Saturday,False,Fair
3,04b2ab83-5325-4dd0-9a46-6d767439f5a2,CU000869,09059923201,UNKNOWN,Voice,4610.0,4734.73,-113.0,2024-05-24 14:01:58,False,...,3G,True,True,True,76.83,5,14,Friday,True,Poor
4,7775e1be-3638-4d37-8ae9-e91fd5a966e5,CU000518,08091140319,TW0150,Sms,271.0,2995.66,-106.0,2024-04-07 20:54:53,False,...,3G,True,True,True,4.52,4,20,Sunday,False,Poor


## Data Loading

In [69]:
!pip install psycopg2 
!pip install sqlalchemy

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.2 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 2.5 MB/s  0:00:00
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 2.4 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.5 MB/s  0:00:00
Using cached typing_extensions-4.14.1-py3-none-any.whl (43 kB)

   ------------- -------------------------- 1/3 [greenlet]
   -----------

In [71]:
pip install python-dotenv

  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)
Note: you may need to restart the kernel to use updated packages.


In [72]:
import psycopg2
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [73]:
load_dotenv()

True

In [74]:
# save to csv
output_file = r'C:\Users\Iyanujesu\Downloads\MTN_pandas_data_engineering\data\processed\call_detail_records.csv'
df_clean.to_csv(output_file, index=False)

In [77]:
# get variables from environment
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# create connection url
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

# load to database
df_clean.to_sql('call_detail_records', con=engine, if_exists='replace', index=False)

print('Data loaded successfully')


Data loaded successfully


In [78]:
with engine.connect() as conn:
    df_clean.to_sql('call_detail_records', con=engine, if_exists='replace', index=False)
    print('Data loaded successfully')

    


Data loaded successfully
